In [1]:
import os
import pandas as pd
import toml
from openpyxl.styles import Font, Border, Side

In [2]:
# Specify config 
config_path = "../configs/config_2024.toml"
config = toml.load(config_path)

# NOTE: only included CO2e in standard summary. N20 and methane are available in upstream data if needed.
# Ideally we just provide them a breakdown of CO2/N20/Methane within CO2e to apply to simplify results
pollutant_list = ["CO2 Equivalent"]   # set pollutant_list = config["pollutant_list"] if full list of pollutants required; this may require additional formatting edits

# Excel formatting
no_border = Border(
    left=Side(border_style=None),
    right=Side(border_style=None),
    top=Side(border_style=None),
    bottom=Side(border_style=None)
)

In [3]:
def process_results(county, analysis_year):  

    df_running = pd.read_csv(os.path.join(config["output_root"], f'data/county/{county}/{analysis_year}/running_summary.csv'))
    df_start = pd.read_csv(os.path.join(config["output_root"], f'data/county//{county}/{analysis_year}/start_summary.csv'))

    # Relabel vehicle types for clarity
    df_running['veh_type'] = df_running["veh_type"].map(config["veh_type_map"])
    df_start['veh_type'] = df_start["veh_type"].map(config["veh_type_map"])

    df_pollutant_running = df_running[df_running['pollutant_name'].isin(pollutant_list)].groupby(['veh_type',"pollutant_name"]).sum()[['running_daily_tons']]
    df_pollutant_running.rename(columns={'running_daily_tons': analysis_year}, inplace=True)
    
    df_pollutant_start = df_start[df_start['pollutant_name'].isin(pollutant_list)].groupby(['veh_type',"pollutant_name"]).sum()[['start_tons']]
    df_pollutant_start.rename(columns={'start_tons': analysis_year}, inplace=True)

    # VMT is the same for all pollutants; select the first one and report 
    df_vmt = df_running[df_running['pollutant_name']==pollutant_list[0]].groupby(['veh_type']).sum()[['daily_vmt_total']]
    df_vmt.rename(columns={'daily_vmt_total': analysis_year}, inplace=True)

    return df_pollutant_running, df_pollutant_start, df_vmt

In [ ]:
county = "King"

running_df = pd.DataFrame()
start_df = pd.DataFrame()
vmt_df = pd.DataFrame()

for year in config["analysis_year_list"]:
    df_year_running_df, df_year_start_df, df_year_vmt_df = process_results(county, year)

    if len(running_df>0): 
        running_df = running_df.merge(df_year_running_df, on=["veh_type","pollutant_name"])
        start_df = start_df.merge(df_year_start_df, on=["veh_type","pollutant_name"])
        vmt_df = vmt_df.merge(df_year_vmt_df, left_index=True, right_index=True)
    else:
        running_df = df_year_running_df.copy()
        start_df = df_year_start_df.copy()
        vmt_df = df_year_vmt_df.copy()

# Total Emissions
total_df = running_df.merge(start_df, on=["veh_type","pollutant_name"], suffixes=["_running", "_start"])
for year in config["analysis_year_list"]:
    total_df[year] = total_df[f"{year}_running"] + total_df[f"{year}_start"]
total_df = total_df[config["analysis_year_list"]]

# Sort by vehicle type order and pollutant name
veh_type_order = ["Passenger Vehicles", "Medium Trucks", "Heavy Trucks", "Bus"]
total_df = total_df.sort_values(by=["veh_type", "pollutant_name"], 
                                key=lambda x: x.map({v: i for i, v in enumerate(config["pollutant_list"])} if x.name == "pollutant_name" else {v: i for i, v in enumerate(veh_type_order)}))
running_df = running_df.sort_values(by=["veh_type", "pollutant_name"], 
                                    key=lambda x: x.map({v: i for i, v in enumerate(config["pollutant_list"])} if x.name == "pollutant_name" else {v: i for i, v in enumerate(veh_type_order)}))
start_df = start_df.sort_values(by=["veh_type", "pollutant_name"], 
                                key=lambda x: x.map({v: i for i, v in enumerate(config["pollutant_list"])} if x.name == "pollutant_name" else {v: i for i, v in enumerate(veh_type_order)}))
vmt_df = vmt_df.sort_values(by=["veh_type"], 
                            key=lambda x: x.map({v: i for i, v in enumerate(veh_type_order)}))

total_df.index.names = ["Vehicle Type", "Pollutant"]
running_df.index.names = ["Vehicle Type", "Pollutant"]
start_df.index.names = ["Vehicle Type", "Pollutant"]
vmt_df.index.names = ["Vehicle Type"]

with pd.ExcelWriter(os.path.join(config["final_summary_dir"],f'county_summary_{county}.xlsx')) as writer:
    total_df.to_excel(writer, sheet_name='Total Emissions')
    running_df.to_excel(writer, sheet_name='Running Emissions')
    start_df.to_excel(writer, sheet_name='Start Emissions')
    vmt_df.to_excel(writer, sheet_name='VMT')

    # Apply formatting
    for sheetname in ['Total Emissions', 'Running Emissions', 'Start Emissions', 'VMT']:
        ws = writer.sheets[sheetname]
        for row in ws.iter_rows():
            for cell in row:
                cell.border = no_border
                if isinstance(cell.value, (int, float)):
                    if sheetname == "Start Emissions":
                        cell.number_format = '0.00'
                    else:
                        cell.number_format = '#,##0'
                cell.font = Font(bold=False)

        # Set column widths based on header text length
        for col in ws.columns:
            header = col[0].value
            width = max(len(str(header)) if header else 20, 20) + 2
            ws.column_dimensions[col[0].column_letter].width = width
    

In [5]:
total_df

,,2023,2024
Vehicle Type,Pollutant,,
Passenger Vehicles,CO2 Equivalent,17185.586508,16623.133712
Medium Trucks,CO2 Equivalent,1496.543066,1462.238045
Heavy Trucks,CO2 Equivalent,2720.491615,2684.050782
Bus,CO2 Equivalent,249.955919,249.139479
